In [20]:
import pdfplumber
import re

In [21]:
import pdfplumber

def extract_all_data(pdf_path):
    """Extracts all text from a given PDF file and prints it out."""
    all_text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_text = page.extract_text() or ""
            print(f"--- Page {i+1} ---")
            print(page_text)
            print()
            all_text += page_text + "\n"
    return all_text

if __name__ == "__main__":
    pdf_file = r"C:\Users\SamsonC\Documents\Accounting\Accounting_AR\RN\46511.pdf"  # Replace with your PDF file path
    # testing pdf 1: 46511
    # testing pdf 2: 46859
    data = extract_all_data(pdf_file)
    print("Complete extracted data:")
    print(data)


--- Page 1 ---
Invoice
8395 Jane Street, Suite 202
Vaughan, ON L4K 5Y2
Telephone 905-738-3177
JOB NUMBER DATE INVOICE NO
BILL TO
Zancor Homes (Brooklin East) LP 23006 5/30/2024 46511
221 North Rivermede Road
Concord, Ontario
L4K 3N7
RE: #23006 - New Brooklin, Brooklin TERMS Net 30
DESCRIPTION QTY RATE AMOUNT
To charge for working drawings for units 32-01 to 32-07. See item 1.1.1 7 4,680.00 32,760.00
a) of contract (final 40% of $11,700 each)
It is a pleasure working with you!
H S T: 864255997 $4,258.80
TOTAL
$37,018.80
Business Number: 864255997
G

Complete extracted data:
Invoice
8395 Jane Street, Suite 202
Vaughan, ON L4K 5Y2
Telephone 905-738-3177
JOB NUMBER DATE INVOICE NO
BILL TO
Zancor Homes (Brooklin East) LP 23006 5/30/2024 46511
221 North Rivermede Road
Concord, Ontario
L4K 3N7
RE: #23006 - New Brooklin, Brooklin TERMS Net 30
DESCRIPTION QTY RATE AMOUNT
To charge for working drawings for units 32-01 to 32-07. See item 1.1.1 7 4,680.00 32,760.00
a) of contract (final 40% of $11

In [30]:
# location = r"C:\Users\SamsonC\Documents\Accounting\Accounting_AR\RN\46859.pdf"
# location = r"C:\Users\SamsonC\Documents\Accounting\Accounting_AR\invoice_pdf\8713.pdf"
location = r"C:\Users\SamsonC\Documents\Accounting\Accounting_AR\RN\46511.pdf"



In [33]:
import pdfplumber
import re

def extract_table_data(table_rows, full_text):
    """
    Extracts table data from a list of table rows.
    Each row is expected to have a description, Qty, Rate, and Amount.
    Multi-line descriptions are concatenated.
    HST information is processed separately.
    """
    table_data = []
    i = 0
    while i < len(table_rows):
        row = table_rows[i].strip()
        # Skip rows that start with HST markers.
        if row.upper().startswith("HST") or row.upper().startswith("H S T"):
            i += 1
            continue

        # Match rows where Qty, Rate, and Amount appear at the end.
        match = re.search(r'(.+?)\s+(\d*\.?\d+)\s+([\d,\.]+)\s+([\d,\.]+)-?', row)
        if match:
            description = match.group(1).strip()
            qty = match.group(2).strip()
            rate = match.group(3).strip()
            price = match.group(4).strip()

            # Check if subsequent lines continue the description.
            next_line_index = i + 1
            while next_line_index < len(table_rows):
                next_line = table_rows[next_line_index].strip()
                if (re.search(r'^\d*\.?\d+\s+[\d,\.]+\s+[\d,\.]+-?$', next_line)
                    or next_line.upper().startswith("HST")
                    or next_line.upper().startswith("H S T")):
                    break
                description += " " + next_line
                next_line_index += 1

            table_data.append((description, qty, rate, price))
            i = next_line_index  # Skip the processed lines.
        else:
            i += 1

    # Process HST information separately using the full text.
    hst_match = re.search(r'HST\s+On\s+Sales\s+([\d.]+%)\s+([\d,\.]+)', full_text, re.IGNORECASE)
    if hst_match:
        hst_rate = hst_match.group(1).strip()
        hst_price = hst_match.group(2).strip()
        table_data.append(("HST On Sales", "-", hst_rate, hst_price))

    return table_data

def extract_invoice_data(pdf_path):
    """
    Extracts key invoice data and table items from the PDF and returns a structured dictionary.
    Expected fields include:
      - invoice_number
      - date
      - client_name
      - address
      - unit
      - building
      - city
      - province
      - postal_code
      - agreement_number
      - client_project
      - items (table rows)
    """
    with pdfplumber.open(pdf_path) as pdf:
        full_text = "\n".join(page.extract_text() for page in pdf.pages if page.extract_text())
    lines = full_text.splitlines()

    # Default values
    invoice_number = "Not found"
    date = "Not found"
    client_name = "Not found"
    agreement_number = "Not found"
    extracted_address = "Not found"
    unit = "Not found"
    city = "Not found"
    province = "Not found"
    postal_code = "Not found"
    client_project = "Not found"

    # ------------------------------
    # 1. Extract Client & Header Details from the BILL TO Block
    # ------------------------------
    # Locate the "BILL TO" block.
    bill_to_idx = next((i for i, line in enumerate(lines) if "BILL TO" in line.upper()), None)
    if bill_to_idx is not None:
        # The next line is expected to have client details in this order:
        # client_name, JOB NUMBER, DATE, INVOICE NO
        if bill_to_idx + 1 < len(lines):
            details_line = lines[bill_to_idx + 1].strip()
            tokens = details_line.split()
            if len(tokens) >= 4:
                # The last token is invoice_number, the second-to-last is the date,
                # the third-to-last is the job number (used here as agreement_number),
                # and the rest form the client_name.
                invoice_number = tokens[-1]
                date = tokens[-2]
                agreement_number = tokens[-3]
                client_name = " ".join(tokens[:-3])
        # Extract address: the next line is the full address (which might include a unit)
        if bill_to_idx + 2 < len(lines):
            full_address_line = lines[bill_to_idx + 2].strip()
            # Extract the unit (e.g., "Suite 201") if present.
            unit_match = re.search(r'(?i)(Suite|Unit)\s*\d+', full_address_line)
            if unit_match:
                unit = unit_match.group(0).strip()
            # Assume the street address is the portion before the first comma.
            extracted_address = full_address_line.split(',')[0].strip()
        # Next line: city and province.
        if bill_to_idx + 3 < len(lines):
            city_prov_line = lines[bill_to_idx + 3].strip()
            if ',' in city_prov_line:
                parts = city_prov_line.split(',')
                city = parts[0].strip()
                province = parts[1].strip()
            else:
                city = city_prov_line
        # Next line: postal code.
        if bill_to_idx + 4 < len(lines):
            postal_code = lines[bill_to_idx + 4].strip()

    # ------------------------------
    # 2. Extract Client Project Details from the RE: Line (if present)
    # ------------------------------
    re_match = re.search(r'RE:\s*#\d+\s*-\s*([^-]+?)(?=\s+(?:TERMS|$))', full_text, re.IGNORECASE)
    if re_match:
        client_project = re_match.group(1).strip()

    # ------------------------------
    # 3. Extract Table Data (e.g., items, quantities, rates, amounts)
    # ------------------------------
    table_data = []
    # Locate the table header row by finding a line with the words "DESCRIPTION", "QTY", "RATE", and "AMOUNT"
    table_header_idx = next((i for i, line in enumerate(lines)
                             if "DESCRIPTION" in line.upper() and "QTY" in line.upper() and 
                                "RATE" in line.upper() and "AMOUNT" in line.upper()), None)
    if table_header_idx is not None:
        # Assume table rows begin after the header and end when a line starting with "TOTAL" or "Business Number:" is encountered.
        table_end_idx = next((i for i, line in enumerate(lines[table_header_idx:], start=table_header_idx)
                              if line.upper().startswith("TOTAL") or "Business Number:" in line), len(lines))
        table_rows = lines[table_header_idx + 1: table_end_idx]
        table_data = extract_table_data(table_rows, full_text)

    # ------------------------------
    # 4. Return Structured Data
    # ------------------------------
    return {
        "invoice_number": invoice_number,
        "date": date,
        "client_name": client_name,
        "address": extracted_address if extracted_address else "No address found",
        "unit": unit,
        "building": "N/A",  # Not specifically extracted in this example
        "city": city,
        "province": province,
        "postal_code": postal_code,
        "agreement_number": agreement_number,
        "client_project": client_project,
        "items": table_data
    }

if __name__ == "__main__":
    pdf_path = location  # Replace with the path to your PDF file.
    data = extract_invoice_data(pdf_path)
    print("Extracted Invoice Data:")
    for key, value in data.items():
        print(f"{key}: {value}")


Extracted Invoice Data:
invoice_number: 46511
date: 5/30/2024
client_name: Zancor Homes (Brooklin East) LP
address: 221 North Rivermede Road
unit: Not found
building: N/A
city: Concord
province: Ontario
postal_code: L4K 3N7
agreement_number: 23006
client_project: New Brooklin, Brooklin
items: [('To charge for working drawings for units 32-01 to 32-07. See item 1.1.1 a) of contract (final 40% of $11,700 each) It is a pleasure working with you!', '7', '4,680.00', '32,760.00')]


In [24]:
import pdfplumber
import re


with pdfplumber.open(location) as pdf:
    text = "\n".join(page.extract_text() for page in pdf.pages if page.extract_text())

    print(text)
print("---------------------")
lines = text.split("\n")  

# extract Date
date_match = re.search(r'Date:\s*(\d{4}-\d{2}-\d{2})', text)
date = date_match.group(1) if date_match else "Date not found"
print(f"Date: {date}")

# extract Invoice Number
invoice_match = re.search(r'Invoice No:\s*(\d+)', text)
invoice_number = invoice_match.group(1) if invoice_match else "Invoice No not found"
print(f"Invoice No: {invoice_number}")

# extract Client Name
client_match = re.search(r'Client:\s*([A-Za-z\s.,-]+)', text)
client_name = client_match.group(1).strip() if client_match else "Client not found"
print(f"Client: {client_name}")

# extract Client Address
extracted_address = None

for i, line in enumerate(lines):
    if client_name in line:
        if i + 1 < len(lines):  # ensure there's a next line
            next_line = lines[i + 1].strip()
            # address validation pattern (assumes addresses start with a number)
            if re.match(r"^\d{1,5}\s[\w\s]+[,\s]*$", next_line):
                extracted_address = next_line
                break

print(f"Address: {extracted_address if extracted_address else 'No address found'}")

unit = re.search(r"^\s*Unit\s*\d+", text, re.MULTILINE)           # Strictly match unit lines
if unit:
    unit_line = unit.group(0).strip()
    print("Unit:", unit_line)

city_test = re.search(r"(?m)^\s*[A-Za-z\s]+(?=, ON)", text)
if city_test:
    print("City:", city_test.group(0))
else:
    print("City not matched.")

province_test = re.search(r"[A-Z]{2}", text)
if province_test:
    print("Province:", province_test.group(0))
else:
    print("Province not matched.")

postal_test = re.search(r"[A-Z]\d[A-Z]\s*\d[A-Z]\d", text)
if postal_test:
    print("Postal Code:", postal_test.group(0))
else:
    print("Postal code not matched.")


Invoice
8395 Jane Street, Suite 202
Vaughan, ON L4K 5Y2
Telephone 905-738-3177
JOB NUMBER DATE INVOICE NO
BILL TO
Zancor Homes (Brooklin East) LP 23006 5/30/2024 46511
221 North Rivermede Road
Concord, Ontario
L4K 3N7
RE: #23006 - New Brooklin, Brooklin TERMS Net 30
DESCRIPTION QTY RATE AMOUNT
To charge for working drawings for units 32-01 to 32-07. See item 1.1.1 7 4,680.00 32,760.00
a) of contract (final 40% of $11,700 each)
It is a pleasure working with you!
H S T: 864255997 $4,258.80
TOTAL
$37,018.80
Business Number: 864255997
G
---------------------
Date: Date not found
Invoice No: Invoice No not found
Client: Client not found
Address: No address found
City: Vaughan
Province: ON
Postal Code: L4K 5Y2


In [25]:
text

'Invoice\n8395 Jane Street, Suite 202\nVaughan, ON L4K 5Y2\nTelephone 905-738-3177\nJOB NUMBER DATE INVOICE NO\nBILL TO\nZancor Homes (Brooklin East) LP 23006 5/30/2024 46511\n221 North Rivermede Road\nConcord, Ontario\nL4K 3N7\nRE: #23006 - New Brooklin, Brooklin TERMS Net 30\nDESCRIPTION QTY RATE AMOUNT\nTo charge for working drawings for units 32-01 to 32-07. See item 1.1.1 7 4,680.00 32,760.00\na) of contract (final 40% of $11,700 each)\nIt is a pleasure working with you!\nH S T: 864255997 $4,258.80\nTOTAL\n$37,018.80\nBusiness Number: 864255997\nG'

In [26]:
# text = "Date: 2020-11-30"

# Extract date using regex
date = re.search(r'Date:\s*(\d{4}-\d{2}-\d{2})', text).group(1)
print(f"Date: {date}")


AttributeError: 'NoneType' object has no attribute 'group'

In [11]:
# Extract invoice
invoice_number = re.search(r'Invoice No: (\d+)', text).group(1)
print(f"Invoice No: {invoice_number}")


Invoice No: 8713


In [12]:
# Extract Client
Client = re.search(r'(?:Client|Bill To|Customer):\s*([\w\s&-]+?(?:\s*(?:Inc\.|Ltd\.|LLC|Limited))?)', text)
pattern = re.search(r'Client:\s*([A-Za-z\s.,-]+)', text).group(1)

# Client
print(f"Client: {pattern}")


Client: Deco Homes



In [13]:
import re

print(repr(text))

# Regex to match the client address
street_regex = r'(\d+\s+[A-Za-z]+\s+[A-Za-z]+)'
street_regex = r'(\d+\s+[A-Za-z]+\s+[A-Za-z]+)'

import re

# Sample text
text = """
Date: 2020-11-30
Invoice
Client: Branthaven Marz Inc.
720 Oval Court
Invoice No: 4560
Burlington, On
L7L 6A9 Agreement No: A0224
Terms: Net 30
Re: #A0224 - Casa De Torri Qty Rate Price
To bill for Hosting and Upgrades as per section 3 of the contract: 1 400.00 400.00
November, 2020
HST On Sales 13.00% 52.00
Total (CDN) $452.00
HST 811629252
It's been a pleasure working with you!
"""

# Split the text into lines
lines = text.splitlines()

# Initialize variables to store address components
street_address = None
city = None
province = None
postal_code = None

# Iterate through the lines to find the address
for i, line in enumerate(lines):
    if "Client: Branthaven Marz Inc." in line:
        # The next line should be the street address
        if i + 1 < len(lines):
            street_address = lines[i + 1].strip()
        # The line after that should be the city and province
        if i + 2 < len(lines):
            city_province = lines[i + 2].strip()
            # Split city and province
            if ", " in city_province:
                city, province = city_province.split(", ")
        # The line after that should be the postal code
        if i + 3 < len(lines):
            postal_code = lines[i + 3].strip().split(" ")[0]  # Extract only the postal code

# Print the full address
if street_address and city and province and postal_code:
    print("Full Address:")
    print(f"{street_address}\n{city}, {province}\n{postal_code}")
else:
    print("Address not found.")

'Date: 2023-10-31\nInvoice\nClient: Deco Homes\n570 Applewood Crescent,\nInvoice No: 8713\nUnit 1\nVaughan, ON Agreement No: A0546\nL4K 4B4\nTerms: Net 30\nRe: #A0546 - Honeystone Qty Rate Price\nTo bill for time spent in October 2023 after launch on document 1.5 150.00 225.00\nrevisions as per Schedule "A1" - Fee Schedule of the contract (1.5\nhours)\n- Schedule S - Site Plan\nHST On Sales 13.00% 29.25\nTotal (CDN) $254.25\nHST 811629252\nThank you for your business.'
Address not found.


In [14]:
agreement_number = re.search(r'Agreement No:\s*([A-Z0-9]+)', text).group(1)
print(f"Agreement No: {agreement_number}")


Agreement No: A0224


In [15]:
terms = re.search(r'Terms:\s*([\w\s]+?)(?:\n|$)', text).group(1).strip()
print(f"Terms: {terms}")



Terms: Net 30


In [16]:
total = re.search(r'Total \(CDN\)\s*\$([\d,.]+)', text).group(1)
print(f"Total: ${total}")



Total: $452.00


In [20]:
import re

# Sample text
text = """
Re: #A0224 - Casa De Torri Qty Rate Price
To bill for Hosting and Upgrades as per section 3 of the contract: 1 400.00 400.00
November, 2020
HST On Sales 13.00% 52.00
"""

# Extract header with Agreement No and Client
header_match = re.search(r'Re: #([A-Z0-9]+) - ([A-Za-z\s]+)\s*(Qty\s+Rate\s+Price)', text)
if header_match:
    agreement_number = header_match.group(1).strip()
    client_name = header_match.group(2).strip()
    column_labels = header_match.group(3).strip()
    print(f"Agreement No: {agreement_number}")
    print(f"Client: {client_name}")
    # print(f"Column Labels: {column_labels}")
    print()

    # Remove the header from the text
    # text_without_header = text.replace(header_match.group(0), "").strip()
    # print("\nText without header:")
    # print(text_without_header)
else:
    print("Header not found.")

# Extract table rows
table_row_regex = r'(.+):\s+(\d+)\s+([\d,.]+)\s+([\d,.]+)'
matches = re.findall(table_row_regex, text)

# Print table data
for match in matches:
    description, qty, rate, price = match
    print(f"Description: {description.strip()}")
    print(f"Qty: {qty}")
    print(f"Rate: {rate}")
    print(f"Price: {price}")

# Extract tax information
tax_match = re.search(r'HST On Sales ([\d.]+%) ([\d,.]+)', text)
if tax_match:
    tax_rate = tax_match.group(1)
    tax_amount = tax_match.group(2)
    print(f"Tax Rate: {tax_rate}")
    print(f"Tax Amount: ${tax_amount}")

Agreement No: A0546
Client: Honeystone

Tax Rate: 13.00%
Tax Amount: $29.25


In [1]:
import base64

with open(r"C:\Users\SamsonC\Documents\Accounting\Accounting_AR\RN\46511.pdf", "rb") as f:
    b64_str = base64.b64encode(f.read()).decode("utf-8")
print(b64_str)


...
